# Step 3: Data Cleaning and Feature Engineering
Each issue from the EDA data quality log is fixed below, with a before and after check. The output is saved to `data/processed/`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW, PROC, IMG = Path('../data/raw'), Path('../data/processed'), Path('../images')
IMG.mkdir(exist_ok=True); PROC.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False})
COLOR = '#2E6F95'

def save(fname):
    plt.tight_layout(); plt.savefig(IMG / fname, dpi=120); plt.show()

def load_raw():
    # EPIC_ID is built from client names: drop it on load so it never appears in any output
    terms = pd.read_excel(RAW / '1-insurance_2025_policy_terms.xlsx', sheet_name='result').drop(columns='EPIC_ID')
    demo = pd.read_excel(RAW / '2-insurance_2025_demographics.xlsx')
    return terms, demo

In [2]:
terms, demo = load_raw()
log = []  # cleaning log: (step, rows affected)
n0 = len(terms)

## 3.1 Strip whitespace from text columns

In [3]:
text_cols = ['Term_Status', 'Line_of_Business', 'Province', 'FSA']
before = sum((terms[c] != terms[c].str.strip()).sum() for c in text_cols)
for c in text_cols:
    terms[c] = terms[c].str.strip()
log.append(('Stripped whitespace in text values', before))
print('Values changed:', before)

Values changed: 100314


## 3.2 Standardise province codes

In [4]:
PROVINCE_FIX = {'NF': 'NL', 'MN': 'MB', 'NW': 'NT', '': 'Unknown'}
changed = terms['Province'].isin(PROVINCE_FIX).sum()
terms['Province'] = terms['Province'].replace(PROVINCE_FIX)
log.append(('Standardised province codes', changed))
terms['Province'].value_counts().to_frame().T

Province,ON,AB,NB,NS,BC,NL,SK,MB,PE,Unknown,QC,NT,NU,YT
count,59893,27301,7319,2467,1729,400,341,237,191,105,6,6,3,2


## 3.3 Remove undocumented line of business

In [5]:
bad = terms['Line_of_Business'] == 'FEC'
terms = terms[~bad].copy()
log.append(('Removed undocumented line of business (FEC)', bad.sum()))
print('Rows removed:', bad.sum())

Rows removed: 1


## 3.4 Flag premium outliers
Rows are kept, because their cancellation flags are still valid; the flag lets later analysis exclude them where premium values matter.

In [6]:
terms['premium_outlier'] = ~terms['Term_Premium'].between(50, 20_000)
log.append(('Flagged premium outliers (< $50 or > $20,000)', terms['premium_outlier'].sum()))
terms['premium_outlier'].value_counts()

premium_outlier
False    99626
True       373
Name: count, dtype: int64

## 3.5 Feature engineering

In [7]:
terms['churned'] = ((terms['CAN_REN'] == 1) | (terms['CAN_MT'] == 1)).astype(int)
terms['LOB_Group'] = terms['Line_of_Business'].map({'AUTO': 'Auto', 'MOTO': 'Auto'}).fillna('Property')

tenure = (terms['Term_Effective_Date'] - terms['Client_First_Written']).dt.days / 365.25
log.append(('Set negative tenure to 0', (tenure < 0).sum()))
terms['tenure_years'] = tenure.clip(lower=0)
terms['tenure_band'] = pd.cut(terms['tenure_years'], [-0.01, 1, 3, 5, 10, 100],
                              labels=['<1 yr', '1-3 yrs', '3-5 yrs', '5-10 yrs', '10+ yrs'])

groups = terms.groupby('Client_id')['LOB_Group'].nunique()
terms['bundled'] = terms['Client_id'].map(groups).eq(2).map({True: 'Auto + Property', False: 'Single line'})
terms['premium_quartile'] = pd.qcut(terms['Term_Premium'], 4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
terms['start_year'] = terms['Term_Effective_Date'].dt.year
terms[['churned', 'LOB_Group', 'tenure_years', 'tenure_band', 'bundled', 'premium_quartile']].head()

,churned,LOB_Group,tenure_years,tenure_band,bundled,premium_quartile
0,0,Auto,0.00,<1 yr,Single line,Q4 (highest)
1,1,Auto,6.00,5-10 yrs,Single line,Q3
2,0,Auto,7.00,5-10 yrs,Single line,Q3
3,1,Auto,8.05,5-10 yrs,Single line,Q4 (highest)
4,0,Auto,7.99,5-10 yrs,Single line,Q3


## 3.6 Demographics: convert birth dates to ages, then drop them

In [8]:
auto = terms[terms['LOB_Group'] == 'Auto'].merge(demo, on='Policy_Term_id', how='inner')
age = lambda col: (auto['Term_Effective_Date'] - auto[col]).dt.days / 365.25
auto['youngest_driver_age'] = age('max_birthdate_max')
auto['oldest_driver_age'] = age('min_birthdate_min')
auto = auto.drop(columns=['max_birthdate_max', 'min_birthdate_min', 'min_applicantbirthdate'])
auto = auto.rename(columns={'max_max_of_drivernumber': 'drivers', 'max_max_of_vehiclenumber': 'vehicles',
                            'max_max_of_yearslicensed': 'years_licensed', 'max_tpbi': 'liability_limit',
                            'max_ap': 'comprehensive_deductible', 'max_col': 'collision_deductible'})
invalid_age = ~auto['youngest_driver_age'].between(16, 100)
auto.loc[invalid_age, 'youngest_driver_age'] = None
log.append(('Auto terms joined with demographics', len(auto)))
log.append(('Driver ages outside 16 to 100 set to missing', (invalid_age & auto['drivers'].notna()).sum()))
auto[['youngest_driver_age', 'oldest_driver_age', 'drivers', 'vehicles']].describe().T

,count,mean,std,min,25%,50%,75%,max
youngest_driver_age,"15,320.00",46.36,18.39,16.00,31.78,43.16,60.90,99.34
oldest_driver_age,"15,354.00",52.53,16.66,16.00,39.16,51.92,64.87,100.81
drivers,"15,354.00",1.90,1.02,1.00,1.00,2.00,2.00,10.00
vehicles,"15,354.00",1.67,0.97,1.00,1.00,1.00,2.00,14.00


## 3.7 Validation checks

In [9]:
assert terms['Policy_Term_id'].is_unique
assert terms['churned'].isin([0, 1]).all()
assert terms['tenure_years'].min() >= 0
assert not terms['Province'].isin(['NF', 'MN', 'NW', '']).any()
assert (terms['Line_of_Business'] != 'FEC').all()
print(f'All checks passed. Rows: {n0:,} -> {len(terms):,}')

All checks passed. Rows: 100,000 -> 99,999

## 3.8 Cleaning log and save

In [10]:
pd.DataFrame(log, columns=['Step', 'Rows affected'])

,Step,Rows affected
0,Stripped whitespace in text values,100314
1,Standardised province codes,263
2,Removed undocumented line of business (FEC),1
3,"Flagged premium outliers (< $50 or > $20,000)",373
4,Set negative tenure to 0,3
5,Auto terms joined with demographics,15354
6,Driver ages outside 16 to 100 set to missing,34


In [11]:
terms.drop(columns=['FSA']).to_pickle(PROC / 'policy_terms_clean.pkl')
auto.drop(columns=['FSA']).to_pickle(PROC / 'auto_drivers_clean.pkl')
print('Saved:', [p.name for p in PROC.glob('*.pkl')])

Saved: ['auto_drivers_clean.pkl', 'policy_terms_clean.pkl', 'risk_scores.pkl']


`FSA` (partial postal code) is dropped from the processed files because province is enough for this analysis and FSA increases re-identification risk.